# 03 · The Beautiful

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/splevine/clustering-good-bad-beautiful/blob/main/notebooks/03_the_beautiful.ipynb)

**Clustering movies by their posters.**

Same pipeline shape as `01_the_good`, completely different modality. CLIP embeddings on poster images, HDBSCAN alongside **EVoC** (a newer density-tree clusterer worth knowing), and a poster-grid visualization that makes the clusters legible at a glance.

The payoff question: *do posters cluster like overviews?* Where they disagree tells us something interesting about how Hollywood markets vs. how it narrates.

## Setup

In [ ]:
# Colab: uncomment to install
# !pip install -q sentence-transformers umap-learn hdbscan evoc datamapplot pandas pyarrow pillow torch matplotlib

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # so `import viz` works

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

plt.rcParams["figure.dpi"] = 110

## Load movies and resolve poster paths

Posters are downloaded by `scripts/fetch_posters.py` to `data/posters/{tmdb_id}.jpg` at w342 resolution. Drop any movie whose poster didn't download.

In [ ]:
movies = pd.read_parquet("../data/movies.parquet")
poster_dir = Path("../data/posters")
movies["poster_file"] = movies["id"].map(lambda i: poster_dir / f"{i}.jpg")
movies = movies[movies["poster_file"].map(lambda p: p.exists())].reset_index(drop=True)
print(f"{len(movies):,} movies with downloaded posters")

## 1. CLIP embeddings on posters

`clip-ViT-B-32` is 512-D and fast. Same `SentenceTransformer` API we used for text — it just takes PIL images instead of strings. We cache the result so repeat runs are instant.

In [ ]:
from sentence_transformers import SentenceTransformer

EMB_DIR = Path("../embeddings")
EMB_DIR.mkdir(exist_ok=True)
clip_path = EMB_DIR / "posters_clip_b32.npy"

if clip_path.exists():
    X_img = np.load(clip_path)
    print(f"loaded cached CLIP embeddings: {X_img.shape}")
else:
    clip_model = SentenceTransformer("clip-ViT-B-32")
    imgs = [Image.open(p).convert("RGB") for p in movies["poster_file"]]
    X_img = clip_model.encode(imgs, batch_size=32, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    np.save(clip_path, X_img)
    print(f"computed and cached: {X_img.shape}")

## 2. UMAP + HDBSCAN on posters

In [ ]:
import hdbscan
from umap import UMAP

umap_cluster = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=0).fit_transform(X_img)
umap_viz = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=0).fit_transform(X_img)

hdb_labels = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=5, cluster_selection_method="eom").fit_predict(umap_cluster)
movies["hdb_cluster"] = hdb_labels
print(f"HDBSCAN: {hdb_labels.max() + 1} clusters, {(hdb_labels == -1).sum()} noise ({(hdb_labels == -1).mean():.1%})")

## 3. EVoC on posters

EVoC (Embedding Vector Clustering) is a newer density-tree clusterer from the McInnes lab — same author as UMAP and HDBSCAN. It builds its own nearest-neighbor graph and doesn't need a pre-reduced input, though passing reduced coords keeps things comparable.

In [ ]:
from evoc import EVoC

evoc_labels = EVoC(base_min_cluster_size=25, n_neighbors=15, random_state=0).fit_predict(umap_cluster)
movies["evoc_cluster"] = evoc_labels
print(f"EVoC:    {evoc_labels.max() + 1} clusters, {(evoc_labels == -1).sum()} noise ({(evoc_labels == -1).mean():.1%})")

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(hdb_labels, evoc_labels)
nmi = normalized_mutual_info_score(hdb_labels, evoc_labels)
print(f"HDBSCAN vs. EVoC  —  ARI: {ari:.3f}  |  NMI: {nmi:.3f}")
print("(1.0 = identical partitions; lower = the two methods carve up the space differently)")

## 4. Poster clusters vs. overview clusters

Run the text pipeline too (reusing cached embeddings from `01`/`02`), then compare: *does how a movie looks on a poster match how it reads in its overview?*

In [ ]:
text_emb_path = EMB_DIR / "overviews_minilm.npy"
if not text_emb_path.exists():
    raise FileNotFoundError("Run notebook 01_the_good.ipynb first to generate text embeddings.")
X_text = np.load(text_emb_path)

# Only keep rows that survived the poster existence filter and have a text embedding.
all_movies = pd.read_parquet("../data/movies.parquet")
all_movies = all_movies[all_movies["overview"].str.len() > 30].reset_index(drop=True)
id_to_idx = {mid: i for i, mid in enumerate(all_movies["id"].tolist())}
keep = [id_to_idx[mid] for mid in movies["id"] if mid in id_to_idx]
X_text_aligned = X_text[keep]
movies = movies[movies["id"].isin(id_to_idx)].reset_index(drop=True)
print(f"aligned: {X_text_aligned.shape[0]} movies with both overview + poster embeddings")

In [ ]:
umap_text = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=0).fit_transform(X_text_aligned)
text_labels = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=5, cluster_selection_method="eom").fit_predict(umap_text)
movies["text_cluster"] = text_labels

text_noise = (text_labels == -1).mean()
poster_noise = (movies["hdb_cluster"] == -1).mean()
ari_text_poster = adjusted_rand_score(text_labels, movies["hdb_cluster"])
print(f"Text clusters:   {text_labels.max() + 1} (noise {text_noise:.1%})")
print(f"Poster clusters: {movies['hdb_cluster'].max() + 1} (noise {poster_noise:.1%})")
print(f"ARI between text and poster partitions: {ari_text_poster:.3f}")
print("(Low ARI doesn't mean one is wrong — it means plot and poster tell different stories.)")

## 5. The payoff — look at the posters

Numbers are fine, but the whole point of poster clustering is that the results are **visible**. Pull the top posters by popularity for each of the biggest clusters.

In [ ]:
def show_cluster_posters(cluster_col, cluster_id, n_posters=12, cols=6, title_prefix=""):
    in_cluster = movies[movies[cluster_col] == cluster_id].sort_values("vote_count", ascending=False).head(n_posters)
    rows = (len(in_cluster) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.8, rows * 2.7))
    axes = np.atleast_2d(axes)
    for ax, (_, row) in zip(axes.flat, in_cluster.iterrows()):
        try:
            ax.imshow(Image.open(row["poster_file"]))
        except Exception:
            pass
        ax.set_title(row["title"][:28], fontsize=8)
        ax.axis("off")
    for ax in axes.flat[len(in_cluster):]:
        ax.axis("off")
    fig.suptitle(f"{title_prefix}Cluster {cluster_id}  ·  {(movies[cluster_col] == cluster_id).sum()} movies", fontsize=11)
    plt.tight_layout(); plt.show()

# Top 4 biggest non-noise HDBSCAN clusters
top_clusters = movies[movies["hdb_cluster"] != -1]["hdb_cluster"].value_counts().head(4).index.tolist()
for cid in top_clusters:
    show_cluster_posters("hdb_cluster", cid, n_posters=12, cols=6, title_prefix="Poster ")

## 6. Spotlight: where do text and image disagree?

Find movies that land in a large text cluster but end up as noise (or in a small cluster) in poster space — i.e., *their poster doesn't look like their story reads*. Those are the marketing stretches or the genre benders.

In [ ]:
disagree = movies[
    (movies["text_cluster"] != -1) & (movies["hdb_cluster"] == -1)
].sort_values("vote_count", ascending=False).head(15)
print("Movies with a clear textual theme but visually unusual posters:\n")
print(disagree[["title", "release_year", "text_cluster", "genres"]].to_string(index=False))

## 7. Fusion: the third lens

Text and poster clusterings disagree (ARI ≈ 0.035 — essentially uncorrelated). But CLIP's text and image encoders produce vectors in the **same** 512-D space, so we can do something the cross-encoder comparison couldn't: **average them per movie** to get a single embedding that captures both what a film says *and* how it looks.

In [ ]:
# Step 1 — encode the same overviews with CLIP's text tower (same embedding space as the posters).
clip_text_path = EMB_DIR / "overviews_clip_b32.npy"

if clip_text_path.exists():
    X_text_clip = np.load(clip_text_path)
    print(f"loaded cached CLIP-text embeddings: {X_text_clip.shape}")
else:
    clip_model = SentenceTransformer("clip-ViT-B-32")  # same model class as cell 6
    X_text_clip = clip_model.encode(
        movies["overview"].tolist(),
        batch_size=64, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    np.save(clip_text_path, X_text_clip)
    print(f"computed and cached: {X_text_clip.shape}")

print(f"X_text_clip shape: {X_text_clip.shape}")
print(f"X_img       shape: {X_img.shape}")
print("Both live in the same 512-D CLIP space → averaging is meaningful.")

In [ ]:
# Step 2 — fuse: average the two normalized vectors, then re-normalize.
X_fused = (X_text_clip + X_img) / 2.0
X_fused = X_fused / (np.linalg.norm(X_fused, axis=1, keepdims=True) + 1e-12)
print(f"X_fused shape: {X_fused.shape}")

In [ ]:
# Step 3 — same UMAP + HDBSCAN pipeline, on the fused embedding.
umap_fused = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=0).fit_transform(X_fused)
fused_labels = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=5, cluster_selection_method="eom").fit_predict(umap_fused)
movies["fused_cluster"] = fused_labels

n_text  = text_labels.max() + 1
n_image = movies["hdb_cluster"].max() + 1
n_fused = fused_labels.max() + 1
noise_text  = (text_labels == -1).mean()
noise_image = (movies["hdb_cluster"] == -1).mean()
noise_fused = (fused_labels == -1).mean()

print(f"text  (MiniLM):  {n_text:>2} clusters · {noise_text:>5.1%} noise")
print(f"image (CLIP):    {n_image:>2} clusters · {noise_image:>5.1%} noise")
print(f"fused (CLIP):    {n_fused:>2} clusters · {noise_fused:>5.1%} noise")
print()
print(f"ARI · text  vs image: {adjusted_rand_score(text_labels, movies['hdb_cluster']):.3f}")
print(f"ARI · text  vs fused: {adjusted_rand_score(text_labels, fused_labels):.3f}")
print(f"ARI · image vs fused: {adjusted_rand_score(movies['hdb_cluster'], fused_labels):.3f}")

In [ ]:
# Step 4 — does fusion resolve the films where text and image disagreed?
disagree_mask = (text_labels != -1) & (movies["hdb_cluster"] == -1)
n_disagree  = int(disagree_mask.sum())
n_recovered = int((disagree_mask & (fused_labels != -1)).sum())
n_still     = n_disagree - n_recovered

print(f"{n_disagree} films had a clear text theme but noisy image position.")
print(f"  → fusion assigned {n_recovered} ({n_recovered/max(n_disagree,1):.1%}) to a coherent fused cluster")
print(f"  → {n_still} stay as noise even after fusion (genuinely ambiguous)")

# Some examples fusion pulled in
resolved = movies[disagree_mask & (fused_labels != -1)].sort_values("vote_count", ascending=False).head(10)
print("\nExamples fusion pulled into a fused cluster:")
print(resolved[["title", "release_year", "text_cluster", "fused_cluster", "genres"]].to_string(index=False))

**Fusion isn't a winner** — it's a third perspective.

- The fused ARI sits *between* text-only and image-only — neither modality dominates.
- Fusion is **less** noisy than either modality alone — you get more confident assignments by combining the two.
- About 57% of the text-clear / image-noisy films get pulled into a coherent fused cluster. The other ~43% stay noise — those are the films where the two modalities genuinely tell different stories, and that's a *signal* worth surfacing to a stakeholder, not a defect.

If the talk's earlier beat was *disagreement is information*, fusion is the next move: **disagreement is also a substrate for resolution**.

## 8. Interactive datamapplot

Render the poster-cluster space as a self-contained, zoomable HTML artifact. The trick is **hierarchical labels** — datamapplot accepts multiple label arrays and shows different ones at different zoom levels: a few coarse super-themes when you're zoomed out, the natural HDBSCAN clusters when you're zoomed in.

We build the hierarchy by merging poster-cluster centroids via cosine-linkage agglomerative clustering, then label each layer with its dominant genres and a recognizable representative film.

Saves to `../map_posters_demo.html` so it doesn't collide with the site's text map at `../map.html` (which is built by `scripts/label_hierarchy.py`).

In [ ]:
import datamapplot
from sklearn.cluster import AgglomerativeClustering
from collections import Counter

# 1 · Build a 3-level hierarchy from the poster-cluster centroids.
real_ids = sorted(int(c) for c in set(movies["hdb_cluster"]) if c != -1)
centroids = np.vstack([X_img[movies["hdb_cluster"] == c].mean(axis=0) for c in real_ids])
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-12

n_fine = len(real_ids)
n_mid = max(8, n_fine // 2)
n_coarse = max(4, n_fine // 4)
print(f"hierarchy levels:  fine={n_fine}  mid={n_mid}  coarse={n_coarse}")

def merge_to(n_target):
    if n_target >= n_fine:
        return list(range(n_fine))
    return AgglomerativeClustering(
        n_clusters=n_target, metric="cosine", linkage="average"
    ).fit_predict(centroids).tolist()

assignments = {n_fine: list(range(n_fine)), n_mid: merge_to(n_mid), n_coarse: merge_to(n_coarse)}

# 2 · Build a readable label per super-cluster: top genres + representative film.
def label_for(film_idx):
    if not len(film_idx):
        return "—"
    sub = movies.iloc[film_idx]
    genres = [g for gs in sub["genres"] for g in gs]
    top_g = " / ".join(g for g, _ in Counter(genres).most_common(2)) if genres else "—"
    rep = sub.sort_values("vote_count", ascending=False).iloc[0]["title"]
    return f"{top_g} · {rep}"

def per_movie_layer(n_target):
    fine_to_super = dict(zip(real_ids, assignments[n_target]))
    super_label = {}
    for s in set(fine_to_super.values()):
        member_fine = [f for f, ss in fine_to_super.items() if ss == s]
        film_idx = movies.index[movies["hdb_cluster"].isin(member_fine)].tolist()
        super_label[s] = label_for(film_idx)
    return [
        "Unlabelled" if c == -1 or c not in fine_to_super else super_label[fine_to_super[int(c)]]
        for c in movies["hdb_cluster"]
    ]

# Pass layers finest -> coarsest, matching the convention used by scripts/label_hierarchy.py.
layer_labels = [per_movie_layer(n_fine), per_movie_layer(n_mid), per_movie_layer(n_coarse)]

# 3 · Rich hover, click-through to TMDB, year-filter histogram, search.
extra = pd.DataFrame({
    "title":   movies["title"].tolist(),
    "year":    movies["release_year"].fillna(0).astype(int).astype(str).tolist(),
    "genre":   movies["genres"].map(lambda gs: gs[0] if len(gs) else "—").tolist(),
    "rating":  movies["vote_average"].round(1).astype(str).tolist(),
    "tmdb_id": movies["id"].astype(str).tolist(),
})
hover_html = (
    '<div style="font-family: ui-sans-serif, system-ui; padding: 2px 4px;">'
    '<div style="font-weight:600">{title}</div>'
    '<div style="color:#9aa0a6; font-size: 0.9em;">{year} · {genre} · ★ {rating}</div>'
    '</div>'
)

plot = datamapplot.create_interactive_plot(
    umap_viz,
    *layer_labels,
    hover_text=movies["title"].tolist(),
    extra_point_data=extra,
    hover_text_html_template=hover_html,
    enable_search=True,
    search_field="title",
    enable_topic_tree=True,
    cluster_layer_colormaps=True,
    histogram_data=pd.to_datetime(movies["release_year"], format="%Y", errors="coerce"),
    histogram_n_bins=30,
    histogram_settings={
        "histogram_title": "Release year",
        "histogram_bin_fill_color": "#ffb454",
        "histogram_bin_selected_fill_color": "#ffb454",
        "histogram_bin_unselected_fill_color": "#3a3f4d",
    },
    darkmode=True,
    on_click="window.open(`https://www.themoviedb.org/movie/{tmdb_id}`)",
    title="Top 5,000 films · poster clusters (CLIP)",
    sub_title="Zoom for finer themes · hover for details · click to open on TMDB.",
)

out_path = Path("../map_posters_demo.html")
plot.save(str(out_path))
print(f"wrote {out_path} ({out_path.stat().st_size // 1024} KB)")

## 9. Posters *as* the points

datamapplot (and most scatterplot tools) render points as dots. The final step: render each point as the actual movie poster using **Bokeh**'s `image_url` glyph. TMDB serves the posters from its CDN, so we reference URLs rather than redistributing files.

Subsample to 2,000 for browser performance (5,000 is fine visually but slow on first load). Stratify so popular movies aren't underrepresented.

In [ ]:
from bokeh.plotting import figure, output_file, save
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.resources import CDN

def tmdb_url(poster_path, size="w92"):
    return f"https://image.tmdb.org/t/p/{size}{poster_path}"

# Stratified subsample: half from popular movies, half from the long tail
N = 2000
rng = np.random.default_rng(0)
top_q = movies["vote_count"].quantile(0.75)
pop_idx = np.where(movies["vote_count"] >= top_q)[0]
rest_idx = np.where(movies["vote_count"] < top_q)[0]
n_pop = min(len(pop_idx), int(N * 0.5))
idx = np.concatenate([
    rng.choice(pop_idx, size=n_pop, replace=False),
    rng.choice(rest_idx, size=min(N - n_pop, len(rest_idx)), replace=False),
])
rng.shuffle(idx)

df = movies.iloc[idx].reset_index(drop=True)
coords = umap_viz[idx]
x, y = coords[:, 0], coords[:, 1]
poster_w = (x.max() - x.min()) * 0.018
poster_h = poster_w * 1.5

source = ColumnDataSource(dict(
    x=x, y=y,
    url=[tmdb_url(p) for p in df["poster_path"]],
    title=df["title"].tolist(),
    year=df["release_year"].astype("Int64").astype(str).tolist(),
    w=[poster_w] * len(df),
    h=[poster_h] * len(df),
))

p = figure(
    width=1400, height=900,
    tools="pan,wheel_zoom,box_zoom,reset,save",
    active_scroll="wheel_zoom",
    title="Top 2,000 movies as poster constellation · CLIP + UMAP",
    background_fill_color="#0f1117",
    border_fill_color="#0f1117",
    outline_line_color=None,
)
p.image_url(url="url", x="x", y="y", w="w", h="h", anchor="center", source=source)
hover_scatter = p.scatter(x="x", y="y", size=12, alpha=0.0, source=source)
p.add_tools(HoverTool(renderers=[hover_scatter], tooltips=[("title", "@title"), ("year", "@year")]))
p.xaxis.visible = False
p.yaxis.visible = False
p.grid.visible = False
p.title.text_color = "#ffb454"

out = Path("../map_posters.html")
output_file(str(out), title="Movies — poster constellation")
save(p, resources=CDN)
print(f"wrote {out} ({out.stat().st_size // 1024} KB)")

## 10. Take it to 3D — the poster cosmos

3D UMAP of the same CLIP embeddings, rendered as Three.js sprites in the browser. Each point is an actual poster, orbit/pan/zoom controls, hover for the title. Self-contained HTML — loads posters lazily from TMDB's CDN.

In [ ]:
from umap import UMAP
import viz  # make sure `sys.path.insert(0, str(Path.cwd()))` was run if needed

umap_3d = UMAP(n_components=3, n_neighbors=15, min_dist=0.2, metric="cosine", random_state=0).fit_transform(X_img)

# Subsample to 1000 stratified by popularity
N = 1000
rng = np.random.default_rng(0)
top_q = movies["vote_count"].quantile(0.5)
pop_idx = np.where(movies["vote_count"] >= top_q)[0]
rest_idx = np.where(movies["vote_count"] < top_q)[0]
n_pop = min(len(pop_idx), int(N * 0.6))
idx_3d = np.concatenate([
    rng.choice(pop_idx, size=n_pop, replace=False),
    rng.choice(rest_idx, size=min(N - n_pop, len(rest_idx)), replace=False),
])
rng.shuffle(idx_3d)

df3 = movies.iloc[idx_3d].reset_index(drop=True)
out = viz.render_poster_cosmos(
    umap_3d[idx_3d],
    [tmdb_url(p) for p in df3["poster_path"]],
    df3["title"].tolist(),
    out_path="../cosmos.html",
    title="3D poster cosmos · 1,000 films",
)
print(f"wrote {out} ({out.stat().st_size // 1024} KB)")

## The Beautiful takeaways

- The same pipeline shape (**embed → reduce → cluster → label → visualize**) works across text and images — you swap the encoder, nothing else.
- Different algorithms carve the same space differently. HDBSCAN and EVoC disagree in specific, interpretable ways; that disagreement is information, not failure.
- Text and image clusters tell different stories about the same movies. The *disagreements* are the richest material for a next conversation with stakeholders.
- **Fused embeddings are a third lens, not a winner.** Averaging CLIP-text and CLIP-image vectors per movie pulls a majority of disagreement films into coherent clusters while leaving the truly ambiguous ones as noise.
- Showing real artifacts — poster thumbnails, not colored dots — changes how people engage with cluster output.